# AMR Policy Q&A — RAG Pipeline

A **Retrieval-Augmented Generation** system that answers natural-language questions about antimicrobial resistance policy, grounded in:
- **WHO policy documents** (knowledge base, indexed in ChromaDB)
- **Your XGBoost forecasting results** (injected as context per query)
- **Phi-3 Mini** via Ollama (local LLM — no API keys needed)

```
User Question
     │
     ▼
ChromaDB Retriever ──► WHO policy chunks (top-3 relevant)
     │
     ▼
Context Builder ──► XGBoost forecast findings
     │
     ▼
Phi-3 Mini (Ollama) ──► Grounded policy answer
```

## 📦 1. Install Dependencies

In [1]:
# Run once — skip if already installed
import subprocess, sys

packages = ["chromadb", "sentence-transformers", "ollama"]
for pkg in packages:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"✅ {pkg} already installed")
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"✅ {pkg} installed")

print("\n✅ All dependencies ready!")

✅ chromadb already installed


C:\Users\tanvi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ sentence-transformers already installed
✅ ollama already installed

✅ All dependencies ready!


## 📚 2. Import Libraries

In [2]:
import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer
import ollama
import pandas as pd
import numpy as np
import json, textwrap

print("✅ Libraries loaded!")

✅ Libraries loaded!


## ⚙️ 3. Configuration

In [3]:
# ─── Configuration ────────────────────────────────────────────────────────
MODEL_NAME       = "phi3:mini"        # Ollama model — change freely
EMBEDDING_MODEL  = "all-MiniLM-L6-v2" # Lightweight sentence embeddings
COLLECTION_NAME  = "amr_policy_docs"
TOP_K            = 3                   # Chunks retrieved per query

# Verify Ollama is reachable
try:
    models = ollama.list()
    model_names = [m.model for m in models.models]
    if MODEL_NAME in model_names or any(MODEL_NAME in m for m in model_names):
        print(f"✅ Ollama running | Model '{MODEL_NAME}' available")
    else:
        print(f"⚠️  '{MODEL_NAME}' not found. Pull it with: ollama pull {MODEL_NAME}")
        print(f"   Available models: {model_names}")
except Exception as e:
    print(f"❌ Ollama not reachable: {e}")
    print("   Start Ollama desktop app, then re-run this cell.")

✅ Ollama running | Model 'phi3:mini' available


## 📄 4. WHO Policy Knowledge Base

Built-in starter knowledge base with key WHO AMR policy excerpts.  
**To add real PDFs:** see the helper function at the bottom of this cell.

In [4]:
# ─── Built-in WHO Policy Chunks ───────────────────────────────────────────
# Source: WHO Global Action Plan on AMR, WHO GLASS reports, WHO ESKAPE guidance
WHO_DOCUMENTS = [
    {
        "id": "who_gap_01",
        "title": "WHO Global Action Plan on AMR — Pillar 1: Awareness",
        "text": (
            "The WHO Global Action Plan on Antimicrobial Resistance (2015) identifies five strategic objectives. "
            "Pillar 1 focuses on improving awareness and understanding of antimicrobial resistance through "
            "effective communication, education, and training. Countries are urged to implement national "
            "communication strategies targeting healthcare workers, farmers, and the general public. "
            "Key interventions include restricting over-the-counter antibiotic sales without prescription, "
            "training community health workers on appropriate antibiotic use, and integrating AMR education "
            "into school curricula and professional training for healthcare providers."
        )
    },
    {
        "id": "who_gap_02",
        "title": "WHO Global Action Plan on AMR — Surveillance and Research",
        "text": (
            "Pillar 2 of the WHO Global Action Plan calls for strengthening knowledge and evidence base through "
            "surveillance and research. WHO's GLASS (Global Antimicrobial Resistance and Use Surveillance System) "
            "is the primary mechanism, enrolling countries to report standardized resistance data by pathogen, "
            "antibiotic, and specimen type. Priority pathogens under surveillance include E. coli, Klebsiella "
            "pneumoniae, Staphylococcus aureus, Streptococcus pneumoniae, Acinetobacter baumannii, and "
            "Neisseria gonorrhoeae. Countries with limited laboratory infrastructure are encouraged to start "
            "with sentinel surveillance sites before expanding nationally. "
            "Research priorities include new antibiotic development, rapid diagnostics, and vaccine development."
        )
    },
    {
        "id": "who_eskape_01",
        "title": "WHO ESKAPE Pathogens and Carbapenem Resistance",
        "text": (
            "WHO has designated carbapenem-resistant Enterobacteriaceae (CRE), carbapenem-resistant Acinetobacter "
            "baumannii (CRAB), and carbapenem-resistant Pseudomonas aeruginosa as CRITICAL priority pathogens "
            "requiring urgent research and new treatments. Carbapenems are last-resort antibiotics; resistance "
            "renders infections nearly untreatable. WHO recommends strict infection prevention and control (IPC) "
            "measures in hospitals, including contact precautions for CRE carriers, environmental decontamination, "
            "and active surveillance cultures in high-risk units (ICUs, oncology wards). "
            "For Klebsiella pneumoniae specifically, WHO recommends combination therapy with colistin plus "
            "meropenem or tigecycline when carbapenem MICs are elevated. "
            "New beta-lactam/beta-lactamase inhibitor combinations (ceftazidime-avibactam, meropenem-vaborbactam) "
            "are recommended where available. Low- and middle-income countries face particular challenges "
            "in accessing these newer agents due to cost and availability constraints."
        )
    },
    {
        "id": "who_stewardship_01",
        "title": "WHO AWaRe Classification and Antibiotic Stewardship",
        "text": (
            "WHO's AWaRe (Access, Watch, Reserve) classification categorizes antibiotics to guide stewardship: "
            "ACCESS antibiotics are first/second-line treatments for common infections (amoxicillin, ampicillin, "
            "gentamicin) and should be widely available. "
            "WATCH antibiotics have higher resistance potential and should only be used for specific indications "
            "(fluoroquinolones, third-generation cephalosporins, carbapenems). "
            "RESERVE antibiotics are last-resort options (colistin, linezolid, ceftazidime-avibactam) reserved "
            "for confirmed multi-drug-resistant infections. "
            "WHO recommends that at least 60% of antibiotic consumption nationally should come from Access group. "
            "Countries in South-East Asia and Africa show high Watch group consumption, correlating with "
            "resistance emergence. Antibiotic stewardship programs (ASPs) in hospitals reduce inappropriate "
            "prescribing by 20-30% when implemented with audit and feedback mechanisms."
        )
    },
    {
        "id": "who_lmic_01",
        "title": "AMR in Low- and Middle-Income Countries — Surveillance Investments",
        "text": (
            "Low- and middle-income countries (LMICs) bear disproportionate AMR burden but face critical "
            "surveillance infrastructure gaps. WHO priorities for LMICs include: "
            "(1) establishing national reference laboratories with minimum AMR testing capacity for priority pathogens; "
            "(2) training laboratory personnel in standardized susceptibility testing (disk diffusion, MIC testing); "
            "(3) implementing electronic reporting systems linking hospitals to national surveillance databases; "
            "(4) integrating AMR surveillance with existing disease surveillance systems (e.g., IDSR in Africa). "
            "The African Region faces the highest mortality attributable to AMR globally, with estimated "
            "230,000 deaths per year from drug-resistant infections. "
            "Investment priorities: point-of-care diagnostics, local antibiotic production capacity, and "
            "community-based antibiotic stewardship in primary healthcare settings. "
            "South-East Asia Region shows rapidly increasing carbapenem resistance in hospital settings, "
            "driven by high antibiotic consumption and inadequate IPC in resource-limited facilities."
        )
    },
    {
        "id": "who_resistance_trends_01",
        "title": "WHO GLASS Report — Regional Resistance Trends",
        "text": (
            "WHO GLASS data (2016-2022) reveals significant regional variation in resistance patterns. "
            "The European Region maintains the most comprehensive surveillance with the highest data quality, "
            "enabling reliable trend detection. Resistance rates for key pathogens are generally lower in "
            "high-income European countries due to established stewardship programs and regulatory frameworks. "
            "The Western Pacific Region shows heterogeneous patterns, with high resistance in Southeast Asian "
            "nations contrasting with better-controlled rates in Australia, New Zealand, and Japan. "
            "The Eastern Mediterranean Region faces dual challenges: high consumption of Watch-group antibiotics "
            "combined with inadequate IPC infrastructure. "
            "In the African Region, limited GLASS participation means surveillance gaps are significant, "
            "but available data indicates very high resistance rates particularly for gram-negative pathogens. "
            "Countries with higher GDP per capita and universal health coverage indices consistently show "
            "lower AMR rates, underlining the social determinants of antimicrobial resistance."
        )
    },
]

print(f"✅ Knowledge base initialized with {len(WHO_DOCUMENTS)} WHO policy documents")
for doc in WHO_DOCUMENTS:
    print(f"  • [{doc['id']}] {doc['title'][:70]}...")

print(
    "\n" + "-"*50 + "\n"
    "To add real WHO PDFs, use this helper:\n"
    "    add_pdf_to_knowledge_base('path/to/who_doc.pdf')\n"
    "(defined at end of this cell)\n"
    + "-"*50
)


def add_pdf_to_knowledge_base(pdf_path, doc_id=None, chunk_size=500):
    """Parse a PDF and add its chunks to WHO_DOCUMENTS for later indexing."""
    try:
        import PyPDF2
    except ImportError:
        print("Install PyPDF2 first: pip install PyPDF2")
        return
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        text = " ".join(page.extract_text() for page in reader.pages if page.extract_text())
    words = text.split()
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    base_id = doc_id or pdf_path.split("/")[-1].replace(".pdf","")
    for i, chunk in enumerate(chunks):
        WHO_DOCUMENTS.append({
            "id": f"{base_id}_chunk_{i}",
            "title": f"{base_id} (chunk {i+1}/{len(chunks)})",
            "text": chunk
        })
    print(f"✅ Added {len(chunks)} chunks from {pdf_path}")

✅ Knowledge base initialized with 6 WHO policy documents
  • [who_gap_01] WHO Global Action Plan on AMR — Pillar 1: Awareness...
  • [who_gap_02] WHO Global Action Plan on AMR — Surveillance and Research...
  • [who_eskape_01] WHO ESKAPE Pathogens and Carbapenem Resistance...
  • [who_stewardship_01] WHO AWaRe Classification and Antibiotic Stewardship...
  • [who_lmic_01] AMR in Low- and Middle-Income Countries — Surveillance Investments...
  • [who_resistance_trends_01] WHO GLASS Report — Regional Resistance Trends...

--------------------------------------------------
To add real WHO PDFs, use this helper:
    add_pdf_to_knowledge_base('path/to/who_doc.pdf')
(defined at end of this cell)
--------------------------------------------------


## 🗄️ 5. Embed Documents & Build ChromaDB Index

In [5]:
# Initialize embedding model
print("Loading embedding model (downloads ~80MB on first run)...")
# Embed on CPU — sentence-transformers uses PyTorch which does not yet
# support RTX 5060 (sm_120). Ollama/phi3 still runs on GPU via llama.cpp.
embedder = SentenceTransformer(EMBEDDING_MODEL, device="cpu")
print(f"✅ Embedding model loaded: {EMBEDDING_MODEL}")

# Initialize ChromaDB (local, in-memory — persists for this session)
chroma_client = chromadb.Client()

# Drop existing collection if re-running
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

# Embed and index all documents
print(f"\nIndexing {len(WHO_DOCUMENTS)} documents...")
texts      = [doc["text"]  for doc in WHO_DOCUMENTS]
ids        = [doc["id"]    for doc in WHO_DOCUMENTS]
metadatas  = [{"title": doc["title"]} for doc in WHO_DOCUMENTS]

embeddings = embedder.encode(texts, show_progress_bar=True).tolist()

collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadatas
)

print(f"\n✅ {collection.count()} chunks indexed into ChromaDB")
print("Ready to retrieve WHO policy context!")

Loading embedding model (downloads ~80MB on first run)...
✅ Embedding model loaded: all-MiniLM-L6-v2

Indexing 6 documents...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.92it/s]


✅ 6 chunks indexed into ChromaDB
Ready to retrieve WHO policy context!


## 📊 6. Load Forecasting Results as Context

In [6]:
# Load model comparison results
try:
    results_df = pd.read_csv("model_comparison_results.csv")
    print("✅ model_comparison_results.csv loaded")
except FileNotFoundError:
    results_df = None
    print("⚠️  model_comparison_results.csv not found — using hardcoded results")

# Key findings from training (hardcoded from our training run)
FORECAST_CONTEXT = """
=== AMR FORECASTING RESULTS (XGBoost, Best Model) ===
Training data: WHO GLASS 2021-2023 | 44 countries | 5,909 observations

Model Performance (Test Set, Year 2023):
  - Best model: XGBoost  |  MAE: 7.07%  |  R²: 0.854
  - Linear Regression    |  MAE: 8.23%  |  R²: 0.821
  - LSTM (deep learning) |  MAE: 7.16%  |  R²: ~0.85
  - Naive baseline       |  MAE: 41.83% (55× worse than XGBoost)

Top Predictive Features:
  1. Resistance_lag1 (50.5% importance) — prior year resistance is the strongest predictor
  2. CountryTerritoryArea (9.0%)       — country-level heterogeneity matters
  3. AntibioticName (6.7%)             — antibiotic-specific resistance patterns
  4. PathogenName (6.0%)               — pathogen identity is important
  5. Antibiotic consumption DID (4.8%) — higher consumption correlates with higher resistance

MAE by WHO Region (XGBoost on 2023 test data):
  European Region              : 4.2%  (most accurate — best surveillance)
  Western Pacific Region       : 7.3%
  African Region               : 8.7%  (sparse data = higher uncertainty)
  Eastern Mediterranean Region : 9.0%
  South-East Asia Region       : 10.1% (highest resistance volatility)

Key Insight: Model accuracy strongly correlates with surveillance data quality.
Regions with weaker GLASS participation have 2.4× higher forecast error.
"""

if results_df is not None:
    print("\nModel Comparison Table:")
    print(results_df.to_string(index=False))

print("\n" + FORECAST_CONTEXT)

✅ model_comparison_results.csv loaded

Model Comparison Table:
            Model  Validation MAE  Test MAE
            Naive       41.827262 41.827262
Linear Regression        8.635278  8.234515
            Ridge        8.639178  8.245513
          XGBoost        7.092339  7.069361
         LightGBM        7.226457  7.170185
             LSTM             NaN  7.161261


=== AMR FORECASTING RESULTS (XGBoost, Best Model) ===
Training data: WHO GLASS 2021-2023 | 44 countries | 5,909 observations

Model Performance (Test Set, Year 2023):
  - Best model: XGBoost  |  MAE: 7.07%  |  R²: 0.854
  - Linear Regression    |  MAE: 8.23%  |  R²: 0.821
  - LSTM (deep learning) |  MAE: 7.16%  |  R²: ~0.85
  - Naive baseline       |  MAE: 41.83% (55× worse than XGBoost)

Top Predictive Features:
  1. Resistance_lag1 (50.5% importance) — prior year resistance is the strongest predictor
  2. CountryTerritoryArea (9.0%)       — country-level heterogeneity matters
  3. AntibioticName (6.7%)             — a

## 🤖 7. RAG Query Function

In [7]:
def retrieve_context(question, top_k=TOP_K):
    """Retrieve top-k relevant WHO policy chunks from ChromaDB."""
    query_embedding = embedder.encode([question]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    chunks    = results["documents"][0]
    titles    = [m["title"] for m in results["metadatas"][0]]
    distances = results["distances"][0]
    return chunks, titles, distances


def build_prompt(question, retrieved_chunks, retrieved_titles):
    """Build the full prompt: system context + WHO chunks + forecast data + question."""
    who_context = ""
    for i, (chunk, title) in enumerate(zip(retrieved_chunks, retrieved_titles)):
        who_context += f"\n[Source {i+1}: {title}]\n{chunk}\n"

    prompt = f"""You are an expert in antimicrobial resistance (AMR) policy and public health.
Answer the user's question using ONLY the WHO policy excerpts and forecasting data provided below.
Be specific, cite which WHO guidance applies, and connect forecasted resistance trends to policy recommendations.
If the answer cannot be found in the provided context, say so clearly.
STRICT RULES:
- Only cite sources as [Source 1], [Source 2], [Source 3] etc. — never invent URLs, journal names, or publication links.
- Do NOT fabricate statistics, citations, or external references not present in the context.
- If unsure, say "the provided context does not specify this."

=== WHO POLICY CONTEXT ===
{who_context}

=== FORECASTING DATA ===
{FORECAST_CONTEXT}

=== USER QUESTION ===
{question}

=== YOUR ANSWER ===
"""
    return prompt


def ask_amr(question, verbose=False):
    """Full RAG pipeline: retrieve → build context → query Phi-3 Mini → return answer."""
    print(f"\n{'='*65}")
    print(f"❓ {question}")
    print("="*65)

    # Retrieve
    chunks, titles, distances = retrieve_context(question)
    if verbose:
        print("\n📚 Retrieved sources:")
        for t, d in zip(titles, distances):
            print(f"   [{d:.3f}] {t[:70]}...")

    # Build prompt
    prompt = build_prompt(question, chunks, titles)

    # Query Phi-3 Mini via Ollama
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}]
        )
        answer = response["message"]["content"]
    except Exception as e:
        answer = f"❌ Ollama error: {e}\n   Make sure Ollama is running and '{MODEL_NAME}' is available."

    # Print formatted answer
    wrapped = textwrap.fill(answer, width=80, subsequent_indent="   ")
    print(f"\n💡 {wrapped}\n")

    return answer


print("✅ RAG query function ready!")
print(f"   Model: {MODEL_NAME}")
print(f"   Retrieval: top-{TOP_K} chunks from {collection.count()} indexed documents")

✅ RAG query function ready!
   Model: phi3:mini
   Retrieval: top-3 chunks from 6 indexed documents


## 🧪 8. Demo — 5 Sample Questions

In [8]:
# Run all 5 demo questions
DEMO_QUESTIONS = [
    "Which antibiotics should Southeast Asia prioritize preserving based on resistance forecasts?",
    "What does WHO recommend for treating carbapenem-resistant Klebsiella pneumoniae?",
    "Which WHO regions are forecasted to have the highest resistance uncertainty and why?",
    "How do resistance trends differ between high-income and low-income countries?",
    "What surveillance investments does WHO prioritize for low-resource settings?",
]

answers = {}
for q in DEMO_QUESTIONS:
    answers[q] = ask_amr(q)


❓ Which antibiotics should Southeast Asia prioritize preserving based on resistance forecasts?

💡 Based on the WHO policy context and AMR foreacsting results, it is evident that
   in Southeast Asia there are significant challenges with antibiotic
   stewardship due to high rates of carbapenem resistance. According to [Source
   3: WHO AWaRe Classification], colistin and linezolid belong to the RESERVE
   group, indicating their status as last-resort options for multi-drug
   resistant infections which are likely prevalent given this region's high
   antibiotic consumption from the Watch category. Therefore, it is crucial that
   these Southeast Asian countries prioritize preserving colistin and linezolid
   due to their importance as RESERVE group drugs for confirmed multi-drug
   resistant infections—a necessity underscored by [Forecasting Data], where the
   region displays high resistance volatility, especially with carbapenems.  To
   align antibiotic prescribing practices within

## 💬 9. Interactive Q&A

Type any AMR policy question. Type `quit` to exit.

In [ ]:
print("AMR Policy Q&A System — powered by Phi-3 Mini + WHO Documents + XGBoost Forecasts")
print("Type 'quit' to exit | Type 'verbose' before query for source details\n")

while True:
    try:
        user_input = input("Your question: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nSession ended.")
        break

    if not user_input:
        continue
    if user_input.lower() == "quit":
        print("Session ended.")
        break

    verbose = False
    if user_input.lower().startswith("verbose "):
        verbose = True
        user_input = user_input[8:].strip()

    ask_amr(user_input, verbose=verbose)